# ASTERIS8：论文式多曝光训练，160帧与400帧对比

本 Notebook 是更新后的主实验入口。它使用原作者 ASTERIS8 网络和官方初始化，
将16张独立曝光交错为8帧输入/8帧目标，执行时间轴与全局3σ裁剪、论文损失、
八帧时间合成和真正的多曝光盲伪源注入。旧的单图复制评估和 α 混合不再用于本实验。

训练优化器与发布源码一致：AdamW、weight decay 1e-4、学习率 1.5e-4、
CosineAnnealingLR T_max=2,000,000。8 GB GPU 使用 AMP，并仅在反向传播时去掉
损失的公共 1e6 倍数以避免 float16 溢出。

In [1]:
from pathlib import Path
import sys
import pandas as pd
import torch

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "asteris":
    PROJECT_ROOT = PROJECT_ROOT.parents[1]
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from astr_ir.asteris.paper_pipeline import (
    PaperAsterisConfig, prepare_paper_dataset, run_paper_inference, train_paper_model
)
from astr_ir.asteris.paper_evaluation import (
    PaperEvaluationConfig, run_paper_mock_evaluation
)

INPUT_ROOT = PROJECT_ROOT / "data" / "processed" / "background"
DATASET_ROOT = PROJECT_ROOT / "data" / "raw" / "our_dataset"
PROFILES = {
    "160": ("90000002", "90000003"),
    "400": ("90000002", "90000003", "90000004", "90000005_1", "90000005_2"),
}
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
CONFIG = PaperAsterisConfig()
CONFIG

PaperAsterisConfig(model='asteris8', patch_t=8, patch_size=128, f_maps=24, num_refinement_blocks=4, scale_factor=4.0, temporal_sigma=3.0, global_sigma=3.0, mse_select=True, epochs=20, samples_per_sequence=64, validation_samples_per_sequence=16, batch_size=1, learning_rate=0.00015, weight_decay=0.0001, scheduler_t_max=2000000, loss_scale=1000000.0, patience=6, seed=20260825, inference_tile_size=128, inference_overlap=16, amp=True, initialize_from_official=True, official_checkpoint='D:/Astr_IR/Asteris/ASTERIS_THU-main/pth/ASTERIS8_nrcshort/ASTERIS8_nrcshort.pth')

## 1. 数据清单与冻结切分

In [2]:
inventory = []
for profile, sequences in PROFILES.items():
    for sequence in sequences:
        inventory.append({
            "profile": profile,
            "sequence": sequence,
            "raw_fits": len(list((DATASET_ROOT / sequence).glob("*.fits"))),
            "background_fits": len(list((INPUT_ROOT / sequence).glob("background_subtracted_*.fits"))),
        })
pd.DataFrame(inventory)

,profile,sequence,raw_fits,background_fits
0,160,90000002,80,80
1,160,90000003,80,80
2,400,90000002,80,80
3,400,90000003,80,80
4,400,90000004,80,80
5,400,90000005_1,80,80
6,400,90000005_2,80,80


In [3]:
RUN_PREPARE = False
if RUN_PREPARE:
    for profile, sequences in PROFILES.items():
        root = PROJECT_ROOT / "data" / "processed" / f"asteris_paper_{profile}"
        prepare_paper_dataset(INPUT_ROOT, DATASET_ROOT, root, sequences=sequences, config=CONFIG)

## 2. ASTERIS8 正式训练

In [4]:
RUN_TRAINING = False
if RUN_TRAINING:
    for profile in ("160", "400"):
        root = PROJECT_ROOT / "data" / "processed" / f"asteris_paper_{profile}"
        train_paper_model(root, config=CONFIG, device=DEVICE)

## 3. 冻结测试曝光的时间合成

In [5]:
RUN_INFERENCE = False
if RUN_INFERENCE:
    for profile in ("160", "400"):
        root = PROJECT_ROOT / "data" / "processed" / f"asteris_paper_{profile}"
        run_paper_inference(
            INPUT_ROOT, DATASET_ROOT, root, root / "checkpoints" / "best_checkpoint.pt",
            evaluation_sequences=("90000002", "90000003"), device=DEVICE, overwrite=True,
        )

## 4. 真正的多曝光盲伪源评估

In [6]:
RUN_EVALUATION = False
if RUN_EVALUATION:
    for profile in ("160", "400"):
        root = PROJECT_ROOT / "data" / "processed" / f"asteris_paper_{profile}"
        evaluation = PROJECT_ROOT / "data" / "processed" / "evaluation" / f"asteris_paper_{profile}"
        run_paper_mock_evaluation(
            INPUT_ROOT, DATASET_ROOT, root, evaluation,
            root / "checkpoints" / "best_checkpoint.pt", device=DEVICE,
            config=PaperEvaluationConfig(),
        )

## 5. 160帧与400帧配对比较

In [7]:
comparison = PROJECT_ROOT / "data" / "processed" / "evaluation" / "asteris_paper_comparison"
summary_path = comparison / "profile_summary.csv"
metrics_path = comparison / "metrics_160_vs_400.csv"
if summary_path.exists():
    display(pd.read_csv(summary_path))
if metrics_path.exists():
    display(pd.read_csv(metrics_path))

,profile,dataset_frames,training_sequences,best_epoch,best_validation_loss,median_coadd_noise_ratio,coadd_noise_reduction
0,160,160,2,19,11727.590210,0.347099,0.652901
1,400,400,5,9,11656.191345,0.336831,0.663169


,target_snr,completeness_160,completeness_400,purity_160,purity_400,f1_160,f1_400,fp_160,fp_400,false_positives_per_frame_160,false_positives_per_frame_400,median_relative_flux_error_160,median_relative_flux_error_400,completeness_delta_400_minus_160,purity_delta_400_minus_160,f1_delta_400_minus_160,fp_delta_400_minus_160,false_positives_per_frame_delta_400_minus_160,median_relative_flux_error_delta_400_minus_160
0,2.0,0.0000,0.0000,0.000000,0.000000,0.000000,0.000000,16.0,12.0,2.0,1.500,NaN,NaN,0.0000,0.000000,0.000000,-4.0,-0.500,NaN
1,3.0,0.4375,0.4375,0.304348,0.350000,0.358974,0.388889,16.0,13.0,2.0,1.625,0.402273,0.337361,0.0000,0.045652,0.029915,-3.0,-0.375,-0.064912
2,4.0,0.8750,0.6875,0.466667,0.478261,0.608696,0.564103,16.0,12.0,2.0,1.500,0.042922,0.039643,-0.1875,0.011594,-0.044593,-4.0,-0.500,-0.003279
3,5.0,1.0000,1.0000,0.500000,0.571429,0.666667,0.727273,16.0,12.0,2.0,1.500,0.008022,-0.012535,0.0000,0.071429,0.060606,-4.0,-0.500,-0.020557
4,7.0,1.0000,1.0000,0.500000,0.571429,0.666667,0.727273,16.0,12.0,2.0,1.500,0.050087,0.049672,0.0000,0.071429,0.060606,-4.0,-0.500,-0.000416
5,10.0,0.9375,0.9375,0.483871,0.555556,0.638298,0.697674,16.0,12.0,2.0,1.500,0.000083,0.022558,0.0000,0.071685,0.059377,-4.0,-0.500,0.022475
